# IVF+PQ tuner — push Recall@100 toward 1.0

Standalone notebook: does **not** require `run_all.sh`. Run after notebook 01 has produced `data/imagenet1m.h5` and `data/imagenet_base.fvecs`.

Variants:
1. **IVFPQ** — baseline (same code path as `02_ivf_benchmark.ipynb`).
2. **OPQ+IVFPQ** — pre-rotation optimised for the data distribution. Same memory budget as base PQ.
3. **IVFPQ+Refine** — reranks top `k×k_factor` PQ candidates with exact L2; stores raw vectors so RAM ≈ baseline + `4 × n × dim` bytes (≈ 4 GB at n=500 K).
4. **OPQ+IVFPQ+Refine** — combined; usually reaches Recall@100 ≥ 0.99.

Writes `results/<mode>/ivf_pq_tuned.csv`.

In [ ]:
import os, sys, subprocess
from pathlib import Path

HERE = Path.cwd()
SCRIPT = HERE / 'scripts' / 'tune_ivfpq.py'
assert SCRIPT.exists(), f'tuner not found at {SCRIPT}'

# Tweak as needed. Defaults match the rest of the lab (n=500_000) and the
# best baseline PQ config from `results/full/ivf_pq.csv` (nlist=1024, M=128).
N_SWEEP   = int(os.environ.get('LAB_N_SWEEP', 500_000))
QUERY_N   = 10_000
NLIST     = 1024
M         = 128
NBITS     = 8
NPROBE    = '16,64,256,1024'
K_FACTORS = '5,10,20'
VARIANTS  = 'all'   # 'base,opq,refine,opq_refine,all'

cmd = [
    sys.executable, str(SCRIPT),
    '--n', str(N_SWEEP), '--query-n', str(QUERY_N),
    '--nlist', str(NLIST), '--m', str(M), '--nbits', str(NBITS),
    '--nprobe', NPROBE, '--k-factors', K_FACTORS,
    '--variants', VARIANTS,
]
print('running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import pandas as pd
df = pd.read_csv('results/full/ivf_pq_tuned.csv')
print('rows:', len(df))
print('\nBest Recall@100 per variant (highest QPS at each variant\'s best recall):')
best = (df.sort_values(['variant','recall_100','qps'], ascending=[True, False, False])
          .groupby('variant').head(1)
          .sort_values('recall_100', ascending=False))
best[['variant','nprobe','k_factor','recall_100','qps','latency_ms','size_mb','build_s']]

In [ ]:
# Best configuration overall (highest R@100, ties broken by QPS):
winner = df.sort_values(['recall_100','qps'], ascending=[False, False]).iloc[0]
print(f"variant       : {winner['variant']}")
print(f"R@100         : {winner['recall_100']:.4f}")
print(f"R@10 / R@1    : {winner['recall_10']:.4f} / {winner['recall_1']:.4f}")
print(f"QPS           : {winner['qps']:,.0f}")
print(f"latency mean  : {winner['latency_ms']:.3f} ms  ·  p99 {winner['latency_p99_ms']:.3f} ms")
print(f"index size    : {winner['size_mb']:,.1f} MB")
print(f"build         : {winner['build_s']:,.1f} s")
print(f"peak RSS      : {winner['rss_peak_mb']/1024:.2f} GB")
print(f"config        : nlist={int(winner['nlist'])} M={int(winner['M'])} nbits={int(winner['nbits'])} "
      f"nprobe={int(winner['nprobe'])} k_factor={winner['k_factor']} "
      f"opq={bool(winner['opq'])} refine={bool(winner['refine'])}")